## Imports and utility

In [ ]:
from schema import common_mice
from schema.mpanze_paw_tracking_refactor import mpanze_paw_tracking_refactor as pt
from schema.mpanze_exp_refactor import mpanze_exp_refactor as exp
from schema.mpanze_widefield_refactor import mpanze_widefield_refactor as wf
from schema.mpanze_glm import mpanze_glm as glm
import numpy as np
from pathlib import Path
import pandas as pd
from datetime import datetime
from tqdm.autonotebook import tqdm
from util.allen_utils import load_allen, overlay_allen

import warnings
import matplotlib.pyplot as plt
from matplotlib import rc
import seaborn as sns
%matplotlib inline
import cv2

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from statannotations.Annotator import Annotator, PValueFormat

from scipy.ndimage import gaussian_filter1d

base = importr('base')
lme4 = importr('lme4')
emmeans = importr('emmeans')
stats = importr('stats')

# define path for datasets
# p_datasets = Path('~/neurophys_3/r_outputs/datasets/').expanduser()
# p_datasets.mkdir(parents=True, exist_ok=True)
p_figures = Path('~/neurophys_3/r_outputs/figures/figure_3/').expanduser()
p_figures.mkdir(parents=True, exist_ok=True)

In [ ]:
# set font to Arial
rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
# set font sizes to 8 for figures
rc('font', size=8)          # controls default text sizes
rc('axes', titlesize=8)     # fontsize of the axes title
rc('axes', labelsize=8)    # fontsize of the x and y labels
rc('xtick', labelsize=8)    # fontsize of the tick labels
rc('ytick', labelsize=8)    # fontsize of the tick labels
rc('legend', fontsize=8)    # legend fontsize
rc('figure', titlesize=8)  # fontsize of the figure title

# set line width to 1
rc('lines', linewidth=1)

# set dpi to 300 for figures
rc('figure', dpi=300)

# svg font type shenanigans
rc('svg', fonttype='none')

# define conversion factor from inches to cm for convenience
cm = 1/2.54

# color palette for cohorts
group_colors = {'Sham':'#BBBBBB', 'Stroke':'#4477AA', 'Stroke + training':'#AA3377'}

figure properties - Manuscript

In [ ]:
# set font to Arial
rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
# set font sizes to 12 for figures
rc('font', size=12)          # controls default text sizes
rc('axes', titlesize=12)     # fontsize of the axes title
rc('axes', labelsize=12)    # fontsize of the x and y labels
rc('xtick', labelsize=12)    # fontsize of the tick labels
rc('ytick', labelsize=12)    # fontsize of the tick labels
rc('legend', fontsize=10)    # legend fontsize
rc('figure', titlesize=12)  # fontsize of the figure title

# set line width to 1
rc('lines', linewidth=1)

# set dpi to 600 for figures
rc('figure', dpi=600)

# svg font type shenanigans
rc('svg', fonttype='none')

fontsize_small = 10
fontsize_medium = 12
fontsize_large = 14

# define conversion factor from inches to cm for convenience
cm = 1/2.54 * 1.5 # (scale larger for Manuscript)

# color palette for cohorts
group_colors = {'Sham':'#BBBBBB', 'Stroke':'#4477AA', 'Stroke + training':'#AA3377'}

## figure 3B, C

In [ ]:
wf_param_id = 6
model_id = 94
# model_id = 131
frames_map = np.array([-10, -5, 0, 5, 10, 15, 20, 25, 30])

keys = (
    glm.RidgeModelFit2
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group="group")
    & "mouse_id > 40"
    & dict(wf_param_id=wf_param_id, model_id=model_id)
    & "phase='Expert'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "stroke_group != 'Learning'"
).fetch("KEY")
print(f"Found {len(keys)} sessions.")


# get number of grasps
n_grasps = len(
    pt.MovementSegmentation.Epoch.proj()
    * pt.EpochClassification.Epoch.proj("epoch_class")
    * pt.PawRecording.Hand.proj("side")
    & dict(side="ipsi", epoch_class="rewarded")
    & keys
)

# assignment dict for loading predictions
subset_dict = {
    "task_limb_onsets": ["TaskLimbOnsets"],
    "support_limb_onsets": ["SupportLimbOnsets"],
    "valve_opening_events": ["ValveOpeningEvents"],
    "cue_events": ["CueEvents"],
    "task" : ["CueEvents", "ValveOpeningEvents"],
}
# subset_dict = {
#     "task_limb": ["MotorFeaturesIpsi"],
#     "support_limb": ["MotorFeaturesContra"],
#     "valve_opening_events": ["ValveOpeningEvents"],
#     "cue_events": ["CueEvents"],
# }
subset_names = (*subset_dict.keys(), "intercept", "total", "true")

# create big arrays to hold responses
response_maps = {
    subset_name: np.full((n_grasps, len(frames_map), 128, 128), np.nan, dtype=np.float32)
    for subset_name in subset_names
}

N = 0
for key in tqdm(keys):
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")
    
    # load predictions
    predictions = (glm.RidgeModelFit2 & key).load_predictions(subset_dict, as_stack=False, include_true=False)

    # load widefield data
    u, svt, h, w = (wf.ImageProcessing2 & key).load_components()
    predictions["true"] = svt.T.astype(np.float32) * 100
    frame_timestamps = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")
    M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
    handedness = (exp.Handedness & key).fetch1("handedness")

    # get paw data
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    start_times, epoch_ids = (
        pt.MovementSegmentation.Epoch.proj("start_time")
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & dict(**key_ipsi, epoch_class="rewarded")
    ).fetch("start_time", "epoch_id", order_by="epoch_id")

    # iterate over epochs
    for start_time, epoch_id in zip(start_times, epoch_ids):
        # check bounds
        start_frame = np.searchsorted(frame_timestamps, start_time)
        if start_frame + frames_map[-1] >= len(frame_timestamps) or start_frame + frames_map[0] < 0:
            continue

        # iterate over predictions
        for subset_name, Y_subset in predictions.items():
            # extract frames
            y_pred_window = Y_subset[start_frame + frames_map]
            # convert to widefield space
            map_pred_window = (u @ y_pred_window.T).T.reshape(-1, h, w)
            for i in range(map_pred_window.shape[0]):
                map_pred_window[i] = cv2.warpAffine(map_pred_window[i], M, (w, h))
                if handedness == "R":
                    map_pred_window[i] = np.fliplr(map_pred_window[i])
            # accumulate
            response_maps[subset_name][N] = map_pred_window
        # increment counter
        N += 1

# average
for subset_name in response_maps.keys():
    response_maps[subset_name] = np.nanmean(response_maps[subset_name], axis=0)

In [ ]:
# save each array as a .npy file
for subset_name, array in response_maps.items():
    np.save(p_figures / f"{subset_name}_response_map_expert_wf{wf_param_id}_model{model_id}.npy", array)

In [ ]:
# load arrays
wf_param_id = 6
model_id = 89
subset_names = ("task_limb_onsets", "support_limb_onsets", "valve_opening_events", "cue_events", "intercept", "total", "true")
response_maps = {}
for subset in subset_names:
    response_maps[subset] = np.load((p_figures / f"{subset}_response_map_expert_wf{wf_param_id}_model{model_id}.npy").as_posix())


### figure 3C

In [ ]:
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl"
    ]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]


vmin = dict(
    task_limb_onsets=0,
    support_limb_onsets=0,
    valve_opening_events=0,
    cue_events=0,
    intercept=0,
    total=0,
    true=0,
)

vmax = dict(
    task_limb_onsets=0.4,
    support_limb_onsets=0.2,
    valve_opening_events=0.2,
    cue_events=0.1,
    intercept=0.1,
    total=0.8,
    true=0.8,
)

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255

subs_to_title = dict(task_limb_onsets="Task limb", support_limb_onsets="Support limb",
                     valve_opening_events="Reward", cue_events="Cue", intercept="Intercept", total="Total", true="Ground truth")

# n_subsets = len(df_expert_maps)
n_subsets = 7
f, ax = plt.subplots(n_subsets-1, 9, figsize=(14*cm, 2*n_subsets*cm),
    gridspec_kw=dict(wspace=0, hspace=0,left=0, right=1.1, top=0.95, bottom=0.05))

for i, subs in enumerate(["task_limb_onsets", "support_limb_onsets", "valve_opening_events", "cue_events", "total", "true"]):
    ax[i, 4].text(0.5, 1, subs_to_title[subs], ha="center", va="bottom", transform=ax[i,4].transAxes, fontsize=fontsize_medium)
    for j in range(9):
        img = response_maps[subs][j].copy()
        # img = df_expert_maps.loc[subs, 0][j].copy()
        img[mask_combined==0] = np.nan
        imsh = ax[i, j].imshow(img, cmap="viridis", vmin=vmin[subs], vmax=vmax[subs])
        overlay_allen(ax[i, j], areas_to_overlay=areas_to_overlay, show_bregma=False,
        line_kw=dict(color="white", linewidth=0.5, alpha=0.5),res=(128,128))
        ax[i,j].set_xlim(10, 118)
    cbar = plt.colorbar(imsh, ax=ax[i,:], shrink=.6, aspect=7, label="$\Delta F/F$ (%)", pad=0.01)
    cbar.ax.text(0.5, 1.05, f"{vmax[subs]:.1f}", ha="center", va="bottom", fontsize=fontsize_small, transform=cbar.ax.transAxes)
    cbar.ax.text(0.5, -0.05, f"{vmin[subs]:.1f}", ha="center", va="top", fontsize=fontsize_small, transform=cbar.ax.transAxes)
    cbar.set_ticks([])

# add time labels
timestamps = [-0.5, -0.25, 0, 0.25, 0.5, 0.75, 1, 1.25, 1.5]
for j, t in enumerate(timestamps):
    ax[-1,j].text(0.5, 0, f"{t:.2f}s", ha="center", va="top", transform=ax[-1,j].transAxes, fontsize=fontsize_small)
ax[-1, 4].text(0.5, -0.3, "Time from rewarded grasp", ha="center", va="top", transform=ax[-1,4].transAxes, fontsize=fontsize_medium)
f.savefig(p_figures / f"figure_3_response_maps_expert_wf{wf_param_id}_model{model_id}.svg", dpi=600, transparent=True)
plt.show()

### figure 3B

In [ ]:
roi = "MOp_ipsi"

wf_param_id = 6
model_id = 94
frames_window = np.arange(-100, 101) # -5 to 5 seconds

keys = (
    glm.SegmentedPredictions
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group="group")
    & "mouse_id > 40"
    & dict(wf_param_id=wf_param_id, model_id=model_id)
    & "phase='Expert'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "stroke_group != 'Learning'"
).fetch("KEY")
print(f"Found {len(keys)} sessions.")


# get number of grasps
n_grasps = len(
    pt.MovementSegmentation.Epoch.proj()
    * pt.EpochClassification.Epoch.proj("epoch_class")
    * pt.PawRecording.Hand.proj("side")
    & dict(side="ipsi", epoch_class="rewarded")
    & keys
)

# assignment dict for loading predictions
subset_dict = {
    "task_limb_onsets": ["TaskLimbOnsets"],
    "support_limb_onsets": ["SupportLimbOnsets"],
    "valve_opening_events": ["ValveOpeningEvents"],
    "cue_events": ["CueEvents"],
    "task" : ["CueEvents", "ValveOpeningEvents"],
}
subset_names = (*subset_dict.keys(), "intercept", "total", "true")

# create big arrays to hold responses
roi_responses_expert = {
    subset_name: np.full((n_grasps, len(frames_window)), np.nan, dtype=np.float32)
    for subset_name in subset_names
}

N = 0
for key in tqdm(keys):
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")

    # load widefield data
    frame_timestamps = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")

    # get paw data
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    start_times, epoch_ids = (
        pt.MovementSegmentation.Epoch.proj("start_time")
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & dict(**key_ipsi, epoch_class="rewarded")
    ).fetch("start_time", "epoch_id", order_by="epoch_id")

    # get dff for the roi
    dff_pred, names = (
        glm.SegmentedPredictions.ROI
        & dict(**key, roi_id=roi)
    ).fetch("dff", "subset_name")
    # get true dff
    dff_true = (
        wf.AllenSegmentation2.ROI & dict(**key, roi_id=roi)
    ).fetch1("dff") * 100
    # combine into dict
    predictions = {n: dff_pred[i] for i, n in enumerate(names)}
    predictions["true"] = dff_true

    for start_time, epoch_id in zip(start_times, epoch_ids):
        # check bounds
        start_frame = np.searchsorted(frame_timestamps, start_time)
        if start_frame + frames_window[-1] >= len(frame_timestamps) or start_frame + frames_window[0] < 0:
            continue

        # iterate over predictions
        for subset_name, dff_subset in predictions.items():
            # extract frames
            dff_window = dff_subset[start_frame + frames_window]
            # accumulate
            roi_responses_expert[subset_name][N] = dff_window
        # increment counter
        N += 1

# average
roi_responses_mean = {}
roi_responses_sem = {}
for subset_name in roi_responses_expert.keys():
    roi_responses_mean[subset_name] = np.nanmean(roi_responses_expert[subset_name], axis=0)
    roi_responses_sem[subset_name] = np.nanstd(roi_responses_expert[subset_name], axis=0) / np.sqrt(np.sum(~np.isnan(roi_responses_expert[subset_name]), axis=0))
    # roi_responses_expert[subset_name] = np.nanmean(roi_responses_expert[subset_name], axis=0)

In [ ]:
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl"
    ]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255

# plot
# get R values
r2_scores = (glm.AllenSegmentation.ROI & keys & "roi_id='MOp_ipsi'").fetch("r2")
r2_mean = np.nanmean(r2_scores)
r2_std = np.nanstd(r2_scores)
f, ax = plt.subplots(1,1, figsize=(4*cm, 3.5*cm),
                     gridspec_kw=dict(left=0.35, right=1, top=1, bottom=0.15))
t = frames_window / 20  # assuming 20 Hz frame rate
t_frame = (t >= -2.5) & (t <= 2.5)

color_palette = {
    "task_limb_onsets": "tab:blue",
    "support_limb_onsets": "tab:orange",
    "valve_opening_events": "tab:green",
    "cue_events": "tab:red",
    "intercept": "tab:purple",
    "total": "black",
    "true": "gray",
}
for subset_name in roi_responses_mean.keys():
    if subset_name == "task" or subset_name == "intercept":
        continue
    m = roi_responses_mean[subset_name]
    s = roi_responses_sem[subset_name]
    ax.plot(t[t_frame], m[t_frame], label=subset_name, color=color_palette[subset_name])
    ax.fill_between(t[t_frame], m[t_frame] - s[t_frame], m[t_frame] + s[t_frame], color=color_palette[subset_name], alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# add inset axes for overlay
ax_inset = f.add_axes([0.75, 0.75, 0.25, 0.25], zorder=-1)
overlay_allen(ax_inset, areas_to_overlay=areas_to_overlay, areas_to_dot=areas_to_dot, show_bregma=False, res=(128,128),
              line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
ax_inset.set_aspect('equal')
ax_inset.set_ylim(128,0)
# create colored masks
tab10 = plt.get_cmap('tab10', 10)
mask_rgb = np.full((128, 128, 3), 1.0, dtype=np.float32)
roi_for_mask = roi.replace('_ipsi', '_L').replace('_contra', '_R')
mask_roi = masks[area_names == roi_for_mask].squeeze()
mask_rgb[mask_roi > 0] = 0.0
ax_inset.imshow(mask_rgb)
ax_inset.patch.set_visible(False)
ax.set_ylabel("Predicted $\Delta F/F$ (%)")
ax.text(0.03, 0.03, f"$R^2$ = {r2_mean:.2f} ± {r2_std:.2f}", transform=ax.transAxes, fontsize=fontsize_small, va="bottom", ha="left")
ax.patch.set_visible(False)
ax.set_yticks([-0.8, 0, 0.8])
f.savefig(p_figures / f"figure_3_roi_trace_{roi}_expert_wf{wf_param_id}_model{model_id}.svg", dpi=600, transparent=True)
plt.show()

## figure 3D

In [ ]:
wf_param_id = 6
model_prefix = "FullModel\_0\_"
model_name = "FullModel_0"

areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl", "AUDp", "VISpm", "VISli", "AUDd",
    "SSs", "VISpor", "TEa", "VISal", "VISl"
    ]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255

df_reduced = (
    (
        glm.AlignedRidgeModelMap2.proj(r2='r2_map_aligned')
        * glm.RidgeModel.proj("model_name")
        * exp.ExperimentalPhase
        * exp.DaysFromStrokeNorm
        * exp.StrokeGroup.proj(stroke_group="group")
        & "mouse_id > 40"
        & f"wf_param_id={wf_param_id}"
        & f"model_name LIKE '{model_prefix}%'"
        & "phase='Expert'"
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "model_name", "r2"])
    .set_index(["mouse_id", "days_from_stroke_norm", "model_name"])
)
df_full = (
    (
        glm.AlignedRidgeModelMap2.proj(r2='r2_map_aligned')
        * glm.RidgeModel.proj("model_name")
        * exp.ExperimentalPhase
        * exp.DaysFromStrokeNorm
        * exp.StrokeGroup.proj(stroke_group="group")
        & "mouse_id > 40"
        & f"wf_param_id={wf_param_id}"
        & f"model_name='{model_name}'"
        & "phase='Expert'"
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "r2"])
    .set_index(["mouse_id", "days_from_stroke_norm" ])
)
df_reduced["delta_r2"] = df_full["r2"] - df_reduced["r2"]


nanmean = lambda x: np.nanmean(np.stack(x, axis=0), axis=0)
df_reduced_avg = df_reduced.groupby(["model_name"]).agg(nanmean)
display(df_reduced_avg)
df_full_avg = np.nanmean(np.stack([d[0] for d in df_full.to_numpy()], axis=0), axis=0)

axes_order = dict(
    FullModel_0_TaskLimbOnsets=0,
    FullModel_0_SupportLimbOnsets=1,
    FullModel_0_ValveOpeningEvents=2,
    FullModel_0_CueEvents=3
)
vmax = dict(
    FullModel_0_TaskLimbOnsets=0.15,
    FullModel_0_SupportLimbOnsets=0.15,
    FullModel_0_ValveOpeningEvents=0.1,
    FullModel_0_CueEvents=0.05,
)

f, ax = plt.subplots(5, 1, figsize=(6*cm, 12*cm),
                     gridspec_kw=dict(left=-0.1, right=1, top=0.9, bottom=0, hspace=0.1))

for i, (model_name, row) in enumerate(df_reduced_avg.iterrows()):
    if model_name not in axes_order:
        continue
    j = axes_order[model_name]
    map_to_plot = row["delta_r2"].copy()
    map_to_plot[mask_combined==0] = np.nan
    imsh=ax[j].imshow(map_to_plot, vmin=0, vmax=vmax[model_name], cmap="plasma")
    cbar = plt.colorbar(imsh, ax=ax[j], shrink=0.7, pad=0.1, label="$\Delta R^2$", aspect=10)
    cbar.ax.set_yticks([0, vmax[model_name]])
    overlay_allen(ax[j], areas_to_overlay=areas_to_overlay, show_bregma=False,
                  line_kw=dict(color="white", linewidth=0.5, alpha=0.5),res=(128,128))
    # ax[j].set_xlim(10, 118)
map_full = df_full_avg.copy()
map_full[mask_combined==0] = np.nan
imsh = ax[-1].imshow(map_full, vmin=0, vmax=0.4, cmap="plasma")
cbar = plt.colorbar(imsh, ax=ax[-1], shrink=0.7, pad=0.1, label="$R^2$", aspect=10)
cbar.ax.set_yticks([0, 0.4])
overlay_allen(ax[-1], areas_to_overlay=areas_to_overlay, show_bregma=False,
              line_kw=dict(color="white", linewidth=0.5, alpha=0.5),res=(128,128))

ax[0].text(0.5, 1.05, "Task limb movements", fontsize=fontsize_medium, ha="center", va="top", transform=ax[0].transAxes)
ax[1].text(0.5, 1.05, "Support limb movements", fontsize=fontsize_medium, ha="center", va="top", transform=ax[1].transAxes)
ax[2].text(0.5, 1.05, "Reward", fontsize=fontsize_medium, ha="center", va="top", transform=ax[2].transAxes)
ax[3].text(0.5, 1.05, "Auditory cue", fontsize=fontsize_medium, ha="center", va="top", transform=ax[3].transAxes)
ax[4].text(0.5, 1.05, "Full model explained variance", fontsize=fontsize_medium, ha="center", va="top", transform=ax[4].transAxes)
ax[0].text(0.5, 0.97, "Unique explained variance", fontsize=fontsize_medium, ha="center", va="top", transform=f.transFigure)
f.savefig(p_figures / f"figure_3_delta_r2_maps_expert_wf{wf_param_id}.svg", dpi=600, transparent=True)
plt.show()

## figure 4 A,B,C + Supplementary figure 4A

In [ ]:
# fetch post stroke data - maps
wf_param_id = 6
model_id = 94
window_response = np.arange(0, 20)

keys = (
    glm.SegmentedPredictions
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group="group")
    & "mouse_id > 40"
    & dict(wf_param_id=wf_param_id, model_id=model_id)
    & "phase!='Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "stroke_group != 'Learning'"
).fetch("KEY")
print(f"Found {len(keys)} sessions.")

# get number of grasps
n_grasps = len(
    pt.MovementSegmentation.Epoch.proj()
    * pt.EpochClassification.Epoch.proj("epoch_class")
    * pt.PawRecording.Hand.proj("side")
    & dict(side="ipsi", epoch_class="rewarded")
    & keys
)
print(f"Found {n_grasps} grasps.")

# assignment dict for loading predictions
subset_dict = {
    "task" : ["CueEvents", "ValveOpeningEvents"],
    "task_limb_onsets": ["TaskLimbOnsets"],
    "support_limb_onsets": ["SupportLimbOnsets"],
    "valve_opening_events": ["ValveOpeningEvents"],
    "cue_events": ["CueEvents"],
}
subset_names = (*subset_dict.keys(), "intercept", "total")


# pre allocate arrays for roi responses
response_maps_all = {
    subset_name: np.full((n_grasps, 128, 128), np.nan, dtype=np.float32)
    for subset_name in subset_names
}
N = 0
index = []
for key in tqdm(keys, desc='computing maps'):
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")

    # load predictions
    predictions = (glm.RidgeModelFit2 & key).load_predictions(subset_dict, as_stack=False, include_true=False)

    # load widefield data
    u, svt, h, w = (wf.ImageProcessing2 & key).load_components()
    frame_timestamps = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")
    M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
    handedness = (exp.Handedness & key).fetch1("handedness")

    # get paw data
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    start_times, epoch_ids = (
        pt.MovementSegmentation.Epoch.proj("start_time")
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & dict(**key_ipsi, epoch_class="rewarded")
    ).fetch("start_time", "epoch_id", order_by="epoch_id")

    # iterate over epochs
    for start_time, epoch_id in zip(start_times, epoch_ids):
        # check bounds
        start_frame = np.searchsorted(frame_timestamps, start_time)
        if start_frame + window_response[-1] >= len(frame_timestamps) or start_frame + window_response[0] < 0:
            continue

        # iterate over predictions
        for subset_name, Y_subset in predictions.items():
            # extract frames
            y_pred_window = Y_subset[start_frame + window_response].mean(axis=0).reshape(-1, 1)
            # convert to widefield space
            map_pred_window = (u @ y_pred_window).T.reshape(h, w)
            map_pred_window = cv2.warpAffine(map_pred_window, M, (w, h))
            if handedness == "R":
                map_pred_window = np.fliplr(map_pred_window)
            # accumulate
            response_maps_all[subset_name][N] = map_pred_window
        # increment counter
        index.append((key["mouse_id"], d, epoch_id))
        N += 1

index_df = pd.MultiIndex.from_tuples(index, names=["mouse_id", "days_from_stroke_norm", "epoch_id"])

dataframes = {}
for subset_name in subset_names:
    response_maps_to_frame = [[m] for m in response_maps_all[subset_name]]
    del response_maps_all[subset_name]
    dataframes[subset_name] = pd.DataFrame(data=response_maps_to_frame, index=index_df, columns=["response_map"])


In [ ]:
# save each dataframe as a pickle file
for subset_name, df in dataframes.items():
    df.to_pickle(p_figures / f"{subset_name}_response_maps_all_wf{wf_param_id}_model{model_id}.pkl")

In [ ]:
# load dataframes
model_id = 94
wf_param_id = 6
subset_names = ["task_limb_onsets", "support_limb_onsets", "valve_opening_events", "cue_events", "intercept", "task", "total"]
dataframes = {}
for subset_name in subset_names:
    dataframes[subset_name] = pd.read_pickle(p_figures / f"{subset_name}_response_maps_all_wf{wf_param_id}_model{model_id}.pkl")

### roi_responses

In [ ]:
# load dataset of roi responses
wf_param_id = 6
model_id = 94
window_response = np.arange(0, 20)
subset_to_load = "task"

rois = [
    "MOs-lateral", "MOs-medial", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSp-anterior", "RSP-posterior", "VISp", "VIS-medial", "VISa", "VISrl"
]
rois_for_stats = [r + "_contra" for r in rois] + [r + "_ipsi" for r in rois]


keys = (
    glm.SegmentedPredictions
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group="group")
    & "mouse_id > 40"
    & dict(wf_param_id=wf_param_id, model_id=model_id)
    & "phase!='Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "stroke_group != 'Learning'"
).fetch("KEY")
print(f"Found {len(keys)} sessions.")

index = []
rows = []

for key in tqdm(keys, desc='computing roi_responses'):
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")

    # load predictions
    roi_dffs, roi_ids = (glm.SegmentedPredictions.Subset & dict(**key, subset_name=subset_to_load)).load_rois(rois_for_stats)

    # load widefield data
    frame_timestamps = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")

    # get paw data
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    start_times, epoch_ids = (
        pt.MovementSegmentation.Epoch.proj("start_time")
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & dict(**key_ipsi, epoch_class="rewarded")
    ).fetch("start_time", "epoch_id", order_by="epoch_id")

    # iterate over epochs
    for start_time, epoch_id in zip(start_times, epoch_ids):
        # check bounds
        start_frame = np.searchsorted(frame_timestamps, start_time)
        if start_frame + window_response[-1] >= len(frame_timestamps) or start_frame + window_response[0] < 0:
            continue

        # iterate over rois
        row_roi = []
        for roi_dff in roi_dffs:
            # extract frames
            roi_response = roi_dff[start_frame + window_response].mean()
            row_roi.append(roi_response)
        rows.append(row_roi)
        index.append((key["mouse_id"], d, epoch_id))

index_df = pd.MultiIndex.from_tuples(index, names=["mouse_id", "days_from_stroke_norm", "epoch_id"])
dataframe_rois_subset = pd.DataFrame(data=rows, index=index_df, columns=roi_ids)
dataframe_rois_subset

In [ ]:
len(rois_for_stats)

### prepare data for stats

In [ ]:
dataset_info = (
    (
        exp.StrokeGroup.proj(stroke_group='group')
        * exp.ExperimentalPhase
        * exp.DaysFromStrokeNorm
        & keys
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "phase", "stroke_group"])
    # .rename(columns={"days_from_stroke_norm": "day"})
    .set_index(["mouse_id", "days_from_stroke_norm"])        
)
# rename groups
dataset_info["stroke_group"] = dataset_info["stroke_group"].replace({"Rehab": "Stroke + training"})
dataset_info["phase"] = dataset_info["phase"].replace({"Expert":"Pre", "Early":"Post Early", "Late":"Post Late"})
# create categorical variables - they work better with R for stats
dataset_info["p"] = pd.Categorical(dataset_info["phase"], categories=["Pre", "Post Early", "Post Late"], ordered=True)
dataset_info["g"] = pd.Categorical(dataset_info["stroke_group"], categories=["Sham", "Stroke", "Stroke + training"], ordered=True)
# day catergorical variable should be converted to string
dataset_info["d"] = pd.Categorical(dataset_info.index.get_level_values("days_from_stroke_norm").astype(str), categories=[str(d) for d in [-3, -2, -1, 3, 7, 14, 21, 28]], ordered=True)
dataset_info

In [ ]:
# compare pre - post maps
subset_to_plot = "task"
nanmedian = lambda x: np.nanmedian(np.stack(x, axis=0), axis=0)
nanmean = lambda x: np.nanmean(np.stack(x, axis=0), axis=0)
pre_baseline = dataframes[subset_to_plot].query("days_from_stroke_norm<0").groupby("mouse_id")['response_map'].agg(nanmedian)
dataframes[subset_to_plot]["map_baselined"] = dataframes[subset_to_plot]["response_map"] - pre_baseline

dataframe_to_plot = (
    dataframes[subset_to_plot]
    .join(dataset_info)
    .groupby(["p", "g"])
    .agg(dict(map_baselined=nanmean))
)

df_response_rois_for_stats = dataframe_rois_subset.stack().to_frame(name='response')
df_response_rois_for_stats.index.names = ["mouse_id", "days_from_stroke_norm", "epoch_id", "roi"]
df_response_rois_for_stats["y"] = df_response_rois_for_stats["response"] - df_response_rois_for_stats.query("days_from_stroke_norm < 0").groupby(["mouse_id", "roi"])['response'].median()

results = []
# run stats for each roi
for roi, df_roi in tqdm(df_response_rois_for_stats.join(dataset_info).query("p!='Pre'").groupby("roi")):
    with (robjects.default_converter + pandas2ri.converter).context():
        model = lme4.lmer('y ~ 1 + g * p + (1|mouse_id/d)', data=df_roi.reset_index())
        formula = "pairwise ~ g | p"
        emm = emmeans.emmeans(model, stats.formula(formula), adjust="none")
        contrasts = base.summary(emm[1])
        confints = stats.confint(emm[1])
    
    # add confints to dataframe
    contrasts = contrasts.set_index(["contrast", "p"])
    confints = confints.set_index(["contrast", "p"]).filter(["asymp.LCL", "asymp.UCL"])
    contrasts = contrasts.join(confints).reset_index()
    contrasts["roi_name"] = roi
    results.append(contrasts)
from scipy.stats import false_discovery_control
df_results_roi = pd.concat(results, ignore_index=True)
df_results_roi["p_adj"] = false_discovery_control(df_results_roi["p.value"], method='bh')
df_results_roi

In [ ]:
df_results_roi.query("roi_name=='MOp_ipsi' and p=='Post Late'")

In [ ]:
# count number of grasps
df_roi.groupby(["p", "g"]).count()

In [ ]:
# output dataframe for manuscript submission
df_p_value_table = df_results_roi.query("p_adj <= 0.05").copy()
map_p_to_phase = {
    "Post Early":"Early",
    "Post Late":"Late"
}
df_p_value_table["stroke_phase"] = df_p_value_table["p"].map(map_p_to_phase)
contrast_to_name = {
    "Sham - Stroke":"Sham vs. Stroke",
    "Sham - (Stroke + training)":"Sham vs. Stroke + training",
    "Stroke - (Stroke + training)":"Stroke vs. Stroke + training"
}
df_p_value_table["contrast_name"] = df_p_value_table["contrast"].map(contrast_to_name)
def format_p_value(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"
def format_ROI(r):
    return r.replace('_contra', '_R').replace('_ipsi', '_L')

def get_comparison(row):
    return f"({row['roi_formatted']}) {row['stroke_phase']}: {row['contrast_name']}"

df_p_value_table["p_value_formatted"] = df_p_value_table["p_adj"].apply(format_p_value)
df_p_value_table["roi_formatted"] = df_p_value_table["roi_name"].apply(format_ROI)
df_p_value_table["comparison"] = df_p_value_table.apply(get_comparison, axis=1)
df_p_value_table = df_p_value_table[["comparison", "p_value_formatted"]]
df_p_value_table.to_csv(p_figures / f"p_value_table_{subset_to_plot}.csv")
display(df_p_value_table)

### plot maps

In [ ]:
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl",
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255


phase_to_ax={'Post Early':0, 'Post Late':1}
vmin = -0.5
vmax = 0.5
cmap = 'PiYG'
f, ax = plt.subplots(3,2, figsize=(9*cm, 10*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0.15, left=0, right=0.95, top=0.9, bottom=0.05))
for phase, df_phase in dataframe_to_plot.groupby("p"):
    if phase == 'Pre':
        continue
    sham_vs_stroke = df_phase.loc[(phase, "Stroke"), "map_baselined"] - df_phase.loc[(phase, "Sham"), "map_baselined"]
    sham_vs_rehab = df_phase.loc[(phase, "Stroke + training"), "map_baselined"] - df_phase.loc[(phase, "Sham"), "map_baselined"]
    stroke_vs_rehab = df_phase.loc[(phase, "Stroke + training"), "map_baselined"] - df_phase.loc[(phase, "Stroke"), "map_baselined"]
    sham_vs_stroke[mask_combined==0] = np.nan
    sham_vs_rehab[mask_combined==0] = np.nan
    stroke_vs_rehab[mask_combined==0] = np.nan
    # plot
    i = phase_to_ax[phase]
    imsh0 = ax[0, i].imshow(sham_vs_stroke, vmin=vmin, vmax=vmax, cmap=cmap)
    imsh1 = ax[1, i].imshow(sham_vs_rehab, vmin=vmin, vmax=vmax, cmap=cmap)
    imsh2 = ax[2, i].imshow(stroke_vs_rehab, vmin=vmin, vmax=vmax, cmap=cmap)
    # allen
    overlay_allen(ax[0, i], areas_to_overlay=areas_to_overlay,  show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.2))
    overlay_allen(ax[1, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.2))
    overlay_allen(ax[2, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.2))
    # titles
    ax[0,i].set_xlim(10,118)
    ax[1,i].set_xlim(10,118)
    ax[2,i].set_xlim(10,118)

    significant_rois = df_results_roi.query("p==@phase and p_adj<0.05")
    for j, row in significant_rois.iterrows():
        # replace _contra
        # replace _contra with _R and _ipsi with _L
        area_to_star = row['roi_name'].replace('_contra', '_R').replace('_ipsi', '_L')
        # get mask for area
        mask_roi = masks[area_names == area_to_star].squeeze()
        # get coordinates of center of mass
        from scipy.ndimage import center_of_mass
        com = center_of_mass(mask_roi)
        p_value = row['p_adj']
        if p_value < 0.001:
            star = '***'
        elif p_value <= 0.01:
            star = '**'
        elif p_value <= 0.05:
            star = '*'
        # contrast
        if row['contrast'] == "Sham - Stroke":
            ax[0,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')
        elif row['contrast'] == "Sham - (Stroke + training)":
            ax[1,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')
        elif row['contrast'] == "Stroke - (Stroke + training)":
            ax[2,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')

    # print(phase, significant_rois)


# ax[0,1].text(0.5, 0.95, "Stroke - Sham", fontsize=8, ha='center', va='center', transform=ax[0,1].transAxes)
# ax[1,1].text(0.5, 0.95, "(Stroke + training) - Sham", fontsize=8, ha='center', va='center', transform=ax[1,1].transAxes)
# ax[2,1].text(0.5, 0.95, "(Stroke + training) - Stroke", fontsize=8, ha='center', va='center', transform=ax[2,1].transAxes)

# pre, early, late
# ax[0,1].text(0.5, 1.1, "Pre", fontsize=8, ha='center', va='center', transform=ax[0,0].transAxes)
# pre, early, late
ax[-1,0].text(0.5, -0.1, "Early", fontsize=fontsize_medium, ha='center', va='bottom', transform=ax[-1,0].transAxes)
ax[-1,1].text(0.5, -0.1, "Late", fontsize=fontsize_medium, ha='center', va='bottom', transform=ax[-1,1].transAxes)
ax[0,0].text(0.5, 0.95, f"Predicted $\\Delta$F/F differences, {subset_to_plot}", fontsize=fontsize_medium, ha='center', va='top', transform=f.transFigure)


# colorbar text
cbar_0 = plt.colorbar(imsh0, ax=ax[0,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_0.ax.text(0.5, -0.07, "Sham > Stroke", fontsize=fontsize_small, ha='center', va='top', transform=cbar_0.ax.transAxes)
cbar_0.ax.text(0.5, 1.07, "Stroke > Sham", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_0.ax.transAxes)

cbar_1 = plt.colorbar(imsh0, ax=ax[1,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_1.ax.text(0.5, -0.07, "Sham > (Stroke + training)", fontsize=fontsize_small, ha='center', va='top', transform=cbar_1.ax.transAxes)
cbar_1.ax.text(0.5, 1.07, "(Stroke + training) > Sham", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_1.ax.transAxes)

cbar_2 = plt.colorbar(imsh0, ax=ax[2,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_2.ax.text(0.5, -0.07, "Stroke > (Stroke + training)", fontsize=fontsize_small, ha='center', va='top', transform=cbar_2.ax.transAxes)
cbar_2.ax.text(0.5, 1.07, "(Stroke + training) > Stroke", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_2.ax.transAxes)


# f.suptitle(f"{subset_to_load}")
f.savefig(p_figures / f'figure_4_widefield_response_differences_wf{wf_param_id}_model{model_id}_{subset_to_load}.svg', dpi=600, transparent=True)
plt.show()




## Figure 4 D,E,F - correlations

In [ ]:
# subsets to 
df_response_rois_for_stats = dataframe_rois_subset.stack().to_frame(name='response')
df_response_rois_for_stats.index.names = ["mouse_id", "days_from_stroke_norm", "epoch_id", "roi"]
df_response_rois_for_stats["y"] = df_response_rois_for_stats["response"] - df_response_rois_for_stats.query("days_from_stroke_norm < 0").groupby(["mouse_id", "roi"])['response'].median()

In [ ]:
dataset_lesion_size = (
    exp.StrokeVolume
    .fetch(format='frame')
    .reset_index()
    .filter(['mouse_id', 'stroke_volume'])
    .dropna()
    .set_index('mouse_id')
)
dataset_lesion_size

# get performance datasets
p_dataset_performance = Path('~/neurophys_3/r_outputs/datasets/dataset_performance_rates.pkl').expanduser()
dataset_performance = pd.read_pickle(p_dataset_performance)
dataset_performance_rel = dataset_performance - dataset_performance.query("day<0").groupby("mouse_id").median()
dataset_performance_rel = dataset_performance_rel.reset_index().rename(columns={"day":"days_from_stroke_norm"}).set_index(["mouse_id", "days_from_stroke_norm"])
display(dataset_performance_rel)

# get fine motor skill datasets
p_dataset_finemotor = Path('~/neurophys_3/r_outputs/datasets/df_onset_offset_ipsi.pkl').expanduser()
dataset_finemotor = pd.read_pickle(p_dataset_finemotor)
dataset_finemotor_rel = dataset_finemotor - dataset_finemotor.query("day<0").groupby("mouse_id").median()
dataset_finemotor_rel = dataset_finemotor_rel.reset_index().rename(columns={"day":"days_from_stroke_norm"}).set_index(["mouse_id", "days_from_stroke_norm"])
dataset_finemotor_rel

In [ ]:
# subset_name = "task"
roi_for_corr = "MOp_ipsi"
finemotor_for_corr = "rewarded_rate"
xlabel = "Change in reward rate (s$^{-1}$)"


df_response_rois_for_stats = dataframe_rois_subset.stack().to_frame(name='response')
df_response_rois_for_stats.index.names = ["mouse_id", "days_from_stroke_norm", "epoch_id", "roi"]
df_response_rois_for_stats["y"] = df_response_rois_for_stats["response"] - df_response_rois_for_stats.query("days_from_stroke_norm < 0").groupby(["mouse_id", "roi"])['response'].median()

df_response_rois_for_stats
df_corr = (
    df_response_rois_for_stats
    .query("roi==@roi_for_corr")
    .join(dataset_info).groupby(["mouse_id", "days_from_stroke_norm"])
    .agg(dict(y='mean'))
    .join(dataset_info)
    .join(dataset_performance_rel)
    # .filter(["y", finemotor_for_corr, "g", "p"])
    .dropna()
)
df_corr.query("days_from_stroke_norm>0").to_csv(p_figures / f'figure_4_roi_vs_performance_{roi_for_corr}_{finemotor_for_corr}_wf{wf_param_id}_model{model_id}_{subset_to_load}.csv')
display(df_corr)

f, ax = plt.subplots(1, 2, figsize=(5.5*cm, 3*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0, top=0.95, bottom=0.2, left=0.2, right=0.95))
early = df_corr.query("p == 'Post Early'").dropna()
colors_early = [group_colors[g] for g in early["g"]]
from scipy.stats import spearmanr
rho, pval = spearmanr(early[finemotor_for_corr], early['y'])
ax[0].scatter(early[finemotor_for_corr], early['y'], label='Early', c=colors_early, alpha=.7, edgecolors='none')
ax[0].text(0.03, 0.97, f"$\\rho$={rho:.3f}\np={pval:.3f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[0].transAxes)
late = df_corr.query("p == 'Post Late' ").dropna()
colors_late = [group_colors[g] for g in late["g"]]
rho, pval = spearmanr(late[finemotor_for_corr], late['y'])
ax[1].scatter(late[finemotor_for_corr], late['y'], label='Late', c=colors_late, alpha=.7, edgecolors='none')
ax[1].text(0.03, 0.97, f"$\\rho$={rho:.3f}\np={pval:.3f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[1].transAxes)

ax[0].set_title("Early")
ax[1].set_title("Late")
ax[0].set_ylabel("Change in $\\Delta$F/F (%)")
ax[0].set_xlabel(xlabel)

# ax[0].set_xlabel("Lesion volume (mm$^3$)")
f.savefig(p_figures / f'figure_4_roi_vs_performance_{roi_for_corr}_{finemotor_for_corr}_wf{wf_param_id}_model{model_id}_{subset_to_load}.svg', dpi=600, transparent=True)
plt.show()